In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_DIR = Path(r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot")

DATA_DIR = PROJECT_DIR / "Data"
OUTPUT_DIR = PROJECT_DIR / "Tests"

In [3]:
def evaluate():

    results = pd.read_csv(
        OUTPUT_DIR / "reconciliation_results.csv"
    )

    ground_truth = pd.read_csv(
        DATA_DIR / "ground_truth.csv"
    )

    # Combine the reconciliation output
    # with the hidden ground truth
    evaluation = results.merge(
        ground_truth,
        on="payment_id",
        how="left"
    )

    return evaluation


if __name__ == "__main__":

    evaluation = evaluate()

    print("\nEvaluation dataset:\n")
    print(evaluation.head())

    print("\nTotal records:")
    print(len(evaluation))

    print("\nTrue scenarios:")
    print(
        evaluation["true_scenario"].value_counts()
    )


Evaluation dataset:

  payment_id  gross_amount payment_date settlement_id settlement_date    fee  \
0   pay_0001          1500   2026-08-01      set_0001      2026-08-11   30.0   
1   pay_0002          5000   2026-08-02      set_0002      2026-08-09  100.0   
2   pay_0003          1500   2026-08-11      set_0003      2026-08-13   30.0   
3   pay_0004          1000   2026-08-07      set_0004      2026-08-09   20.0   
4   pay_0005         10000   2026-08-02      set_0005      2026-08-09  200.0   

    tax  expected_net_amount  settlement_net_amount  bank_amount  \
0   5.4                  NaN                 1464.6          NaN   
1  18.0                  NaN                 4882.0          NaN   
2   5.4               1464.6                 1464.6       1464.6   
3   3.6                  NaN                  976.4          NaN   
4  36.0                  NaN                 9764.0          NaN   

   refund_amount  delay_days           status               reason  \
0            NaN  

In [4]:
def calculate_accuracy(evaluation):

    evaluation["expected_status"] = (
        evaluation["true_scenario"]
        .apply(
            lambda x:
            "AUTO_RECONCILED"
            if x == "NORMAL"
            else "EXCEPTION"
        )
    )

    evaluation["correct"] = (
        evaluation["status"]
        == evaluation["expected_status"]
    )

    accuracy = evaluation["correct"].mean()

    return accuracy


if __name__ == "__main__":

    evaluation = evaluate()

    accuracy = calculate_accuracy(evaluation)

    print(f"\nBaseline accuracy: {accuracy:.2%}")


Baseline accuracy: 100.00%


In [5]:
    print(OUTPUT_DIR)

C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests


In [6]:
import pandas as pd

output_path = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Output\reconciliation_results.csv"

results = pd.read_csv(output_path)

print("OUTPUT RECONCILIATION FILE")
print("=" * 60)

print(
    results[
        results["payment_id"].isin(
            ["pay_0001", "pay_0004", "pay_0093"]
        )
    ].T
)

OUTPUT RECONCILIATION FILE
                                     0                    3   \
payment_id                     pay_0001             pay_0004   
gross_amount                       1500                 1000   
payment_date                 2026-08-01           2026-08-07   
settlement_id                  set_0001             set_0004   
settlement_date              2026-08-11           2026-08-09   
fee                                30.0                 20.0   
tax                                 5.4                  3.6   
expected_net_amount              1464.6                976.4   
settlement_net_amount            1464.6                976.4   
bank_amount                         NaN                  NaN   
refund_amount                       NaN                  NaN   
delay_days                           10                    2   
status                        EXCEPTION            EXCEPTION   
reason                 SETTLEMENT_DELAY  MISSING_BANK_RECORD   

            